# Binding Predictor v2 — 2025–26 wala upgrade

**v1 (Step 1) me kya tha:** sirf Davis data (68 dawayein) + simple fingerprints → random split pe achha (CI 0.879), lekin **nayi dawaon pe kamzor (CI 0.598)**.

**v2 me kya naya hai:**
| Cheez | v1 | v2 |
|---|---|---|
| Data | Davis: 68 dawayein, 442 proteins, ~30k jode | **Davis + KIBA: ~2,180 dawayein, ~670 proteins, ~148k jode** |
| Protein ki samajh | amino acid ginti | **ESM-2** (Meta ka protein AI, 650M parameters, 2023) |
| Molecule ki samajh | fingerprint | fingerprint + chemistry descriptors + **ChemBERTa** (molecule AI) |
| Test | random + nayi dawa | random + **nayi dawa** + **naya protein** |

Notebook khud **v1 vs v2 ka comparison** karega, taaki saaf dikhe kitna sudhaar hua.

### ⚠️ Pehle GPU on karo (zaroori)
Colab me upar **Runtime → Change runtime type → T4 GPU → Save**. Phir **Runtime → Run all**.
- Pehli baar ~30–50 minute. AI embeddings aur models **Google Drive me save** honge, agli baar tez.
- Free GPU kabhi-kabhi busy hota hai; mile nahi to thodi der baad try karo.

## 1. Tools install + GPU check

In [ ]:
!pip install -q rdkit transformers
import os, json, pickle, time, urllib.request, warnings
from collections import OrderedDict
import numpy as np, pandas as pd
import torch
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings("ignore")
import xgboost as xgb
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import joblib

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
print("GPU mila ✅" if GPU else "⚠️ GPU nahi mila — Runtime → Change runtime type → T4 GPU karo, warna bahut slow chalega")

## 2. Google Drive jodo (sab kuch save karne ke liye)

In [ ]:
USE_DRIVE = True
SAVE_DIR = "saved"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SAVE_DIR = "/content/drive/MyDrive/ai_drug_platform/binding_v2"
    except Exception as e:
        print("Drive nahi juda, local folder use karenge:", e)
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save folder:", SAVE_DIR)

## 3. Data download: Davis + KIBA
- **Davis:** binding strength = pKd (zyada = strong).
- **KIBA:** binding ka combined score (alag scale, isliye dono ke models alag train honge, same tareeke se).

In [ ]:
BASES = ["https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data/{}/",
         "https://raw.githubusercontent.com/thinng/GraphDTA/master/data/{}/"]

def download(ds, f):
    path = os.path.join("data", ds, f)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for b in BASES:
        if os.path.exists(path) and os.path.getsize(path) > 1000:
            break
        try:
            urllib.request.urlretrieve(b.format(ds) + f, path)
        except Exception as e:
            print("✘", b.format(ds) + f, "->", e)
            if os.path.exists(path): os.remove(path)
    assert os.path.exists(path), f"{ds}/{f} download nahi hua — screenshot Claude ko bhejo"
    return path

DATA = {}
for ds in ["davis", "kiba"]:
    lig = json.load(open(download(ds, "ligands_can.txt")), object_pairs_hook=OrderedDict)
    pro = json.load(open(download(ds, "proteins.txt")), object_pairs_hook=OrderedDict)
    Y = np.asarray(pickle.load(open(download(ds, "Y"), "rb"), encoding="latin1"), dtype=float)
    if ds == "davis":
        Y = -np.log10(Y / 1e9)          # Kd (nM) -> pKd
    ii, jj = np.where(~np.isnan(Y))
    pairs = pd.DataFrame({"d": ii, "p": jj, "y": Y[ii, jj].astype(np.float32)})
    DATA[ds] = {"drug_ids": list(lig), "smiles": list(lig.values()),
                "prot_ids": list(pro), "seqs": list(pro.values()), "pairs": pairs}
    print(f"{ds:5s}: {len(lig):,} dawayein · {len(pro):,} proteins · {len(pairs):,} jode")

## 4. AI embeddings banao (ye v2 ka asli dimaag hai)
- **ESM-2** har protein ko 1280 numbers me badalta hai, jo uski 3D shape aur function ka andaza rakhte hain — ye model karodon protein sequences padh ke seekha hai.
- **ChemBERTa** har molecule ko 384 numbers me badalta hai — ye 7.7 crore molecules se seekha hai.

Ye ek baar banenge aur Drive me save ho jayenge.

In [ ]:
from transformers import AutoTokenizer, AutoModel

ESM_NAME = "facebook/esm2_t33_650M_UR50D" if GPU else "facebook/esm2_t12_35M_UR50D"
CHEM_NAME = "DeepChem/ChemBERTa-77M-MTR"

@torch.no_grad()
def embed_proteins(seqs, window=1000):
    tok = AutoTokenizer.from_pretrained(ESM_NAME)
    model = AutoModel.from_pretrained(ESM_NAME).to(DEVICE).eval()
    if GPU: model = model.half()
    out = []
    for k, s in enumerate(seqs):
        chunks = [s[i:i + window] for i in range(0, len(s), window)] or [s]
        vecs = []
        for c in chunks:                       # lambi protein ko tukdon me padho, phir average
            t = tok(c, return_tensors="pt").to(DEVICE)
            h = model(**t).last_hidden_state[0, 1:-1].float()
            vecs.append(h.mean(0).cpu().numpy())
        out.append(np.mean(vecs, 0))
        if (k + 1) % 100 == 0: print(f"  proteins {k+1}/{len(seqs)}")
    del model; torch.cuda.empty_cache() if GPU else None
    return np.stack(out).astype(np.float32)

@torch.no_grad()
def embed_molecules(smiles, batch=64):
    tok = AutoTokenizer.from_pretrained(CHEM_NAME)
    model = AutoModel.from_pretrained(CHEM_NAME).to(DEVICE).eval()
    out = []
    for i in range(0, len(smiles), batch):
        t = tok(smiles[i:i + batch], return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)
        h = model(**t).last_hidden_state
        m = t["attention_mask"].unsqueeze(-1).float()
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    del model; torch.cuda.empty_cache() if GPU else None
    return np.concatenate(out).astype(np.float32)

def cached(name, fn):
    path = os.path.join(SAVE_DIR, name + ".npy")
    if os.path.exists(path):
        print("  Drive se load:", name); return np.load(path)
    t0 = time.time(); arr = fn(); np.save(path, arr)
    print(f"  bana ke save: {name} {arr.shape} ({time.time()-t0:.0f} sec)"); return arr

for ds, D in DATA.items():
    print(ds)
    D["esm"]  = cached(f"{ds}_esm_{ESM_NAME.split('/')[-1]}", lambda: embed_proteins(D["seqs"]))
    D["chem"] = cached(f"{ds}_chemberta", lambda: embed_molecules(D["smiles"]))
print("Embeddings ready ✅")

## 5. Purane (v1) aur naye (v2) features
- **v1:** Morgan fingerprint + amino acid ginti (Step 1 jaisa)
- **v2:** Morgan fingerprint + RDKit descriptors + ChemBERTa + ESM-2

In [ ]:
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
AA = "ACDEFGHIKLMNPQRSTVWY"; AA_IDX = {a: k for k, a in enumerate(AA)}

def aa_feat(seq):
    seq = [a for a in seq.upper() if a in AA_IDX]
    comp = np.zeros(20, np.float32); dip = np.zeros(400, np.float32)
    for a in seq: comp[AA_IDX[a]] += 1
    for a, b in zip(seq, seq[1:]): dip[AA_IDX[a] * 20 + AA_IDX[b]] += 1
    return np.concatenate([comp / max(len(seq), 1), dip / max(len(seq) - 1, 1)])

def mol_feats(smiles):
    fps, descs = [], []
    for s in smiles:
        m = Chem.MolFromSmiles(s)
        if m is None:
            fps.append(np.zeros(1024, np.float32)); descs.append(np.zeros(len(Descriptors._descList), np.float32)); continue
        fps.append(fp_gen.GetFingerprintAsNumPy(m).astype(np.float32))
        d = []
        for _, fn in Descriptors._descList:
            try: d.append(float(fn(m)))
            except Exception: d.append(np.nan)
        descs.append(np.nan_to_num(np.clip(np.array(d, np.float64), -1e6, 1e6)).astype(np.float32))
    return np.stack(fps), np.stack(descs)

for ds, D in DATA.items():
    D["fp"], D["desc"] = mol_feats(D["smiles"])
    D["aa"] = np.stack([aa_feat(s) for s in D["seqs"]])
    D["drug_v1"], D["prot_v1"] = D["fp"], D["aa"]
    D["drug_v2"] = np.hstack([D["fp"], D["desc"], D["chem"]])
    D["prot_v2"] = D["esm"]
    print(f"{ds}: drug v1 {D['drug_v1'].shape[1]} → v2 {D['drug_v2'].shape[1]} features · protein v1 {D['prot_v1'].shape[1]} → v2 {D['prot_v2'].shape[1]}")

def build_X(D, ver, rows):
    pr = D["pairs"].iloc[rows]
    return np.hstack([D[f"drug_{ver}"][pr.d.values], D[f"prot_{ver}"][pr.p.values]]).astype(np.float32)

## 6. Metrics aur 3 tarah ke test
- **Random:** aasan.
- **Nayi dawa (cold drug):** test ki dawayein training me kabhi nahi dikhi.
- **Naya protein (cold target):** test ke proteins training me kabhi nahi dikhe.

In [ ]:
def concordance_index(y_true, y_pred, chunk=2000):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    num = den = 0.0
    for s in range(0, len(y_true), chunk):
        yt, yp = y_true[s:s+chunk, None], y_pred[s:s+chunk, None]
        mask = yt > y_true[None, :]
        diff = yp - y_pred[None, :]
        num += ((diff > 0) & mask).sum() + 0.5 * ((diff == 0) & mask).sum()
        den += mask.sum()
    return num / den if den else float("nan")

def make_model():
    return xgb.XGBRegressor(n_estimators=1000, max_depth=8, learning_rate=0.05, subsample=0.8,
                            colsample_bytree=0.4, tree_method="hist", device=DEVICE,
                            n_jobs=-1, random_state=42)

def splits_for(D):
    n = len(D["pairs"]); idx = np.arange(n)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    return {"random":      train_test_split(idx, test_size=0.2, random_state=42),
            "nayi dawa":   next(gss.split(idx, groups=D["pairs"].d.values)),
            "naya protein": next(gss.split(idx, groups=D["pairs"].p.values))}

## 7. Bada muqabla: v1 vs v2 🥊
2 datasets × 3 tests × 2 versions = 12 models. GPU pe ~15–30 minute.

In [ ]:
rows = []
t0 = time.time()
for ds, D in DATA.items():
    y = D["pairs"].y.values
    for split_name, (tr, te) in splits_for(D).items():
        for ver in ["v1", "v2"]:
            model = make_model().fit(build_X(D, ver, tr), y[tr])
            p = model.predict(build_X(D, ver, te))
            rows.append({"dataset": ds, "test": split_name, "version": ver,
                         "MSE": float(np.mean((y[te] - p) ** 2)),
                         "Pearson": float(pearsonr(y[te], p)[0]),
                         "CI": float(concordance_index(y[te], p))})
            print(f"{ds:5s} | {split_name:12s} | {ver}: CI={rows[-1]['CI']:.3f}  MSE={rows[-1]['MSE']:.3f}  ({time.time()-t0:.0f}s)")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(SAVE_DIR, "v1_vs_v2_results.csv"), index=False)
table = res.pivot_table(index=["dataset", "test"], columns="version", values="CI").round(3)
table["sudhaar"] = (table["v2"] - table["v1"]).round(3)
table

**Kaise padhein:** `CI` jitna zyada utna behtar (0.5 = tukka). `sudhaar` column me plus (+) matlab v2 jeeta.
Sabse zaroori rows: **nayi dawa** aur **naya protein** — asli duniya me yahi kaam aata hai.

**Published reference (approximate, random split):** Davis CI ~0.88–0.90 (DeepDTA/GraphDTA), KIBA CI ~0.86–0.89. Cold splits pe papers me score kaafi girta hai — wahan v1 se behtar hona hi asli jeet hai.

## 8. Graph

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, ds in zip(axes, DATA):
    t = res[res.dataset == ds].pivot(index="test", columns="version", values="CI").loc[["random", "nayi dawa", "naya protein"]]
    t.plot.bar(ax=ax, rot=0, color=["#9aa5b1", "#2f6fdf"])
    ax.axhline(0.5, color="gray", ls="--", lw=1); ax.set_ylim(0.4, 1.0)
    ax.set_title(ds.upper()); ax.set_ylabel("CI (zyada = behtar)"); ax.set_xlabel("")
plt.suptitle("v1 vs v2 — binding prediction"); plt.tight_layout(); plt.show()

## 9. Final v2 models saare data pe train karke Drive me save
Ye models agle step me **demo app v2** me lagenge.

In [ ]:
for ds, D in DATA.items():
    path = os.path.join(SAVE_DIR, f"binding_v2_{ds}.joblib")
    if os.path.exists(path):
        print("Pehle se saved:", path); continue
    allrows = np.arange(len(D["pairs"]))
    model = make_model().fit(build_X(D, "v2", allrows), D["pairs"].y.values)
    joblib.dump({"model": model, "prot_ids": D["prot_ids"], "seqs": D["seqs"],
                 "esm_name": ESM_NAME, "chem_name": CHEM_NAME}, path)
    print("Saved ✅", path)

## Next
**Section 7 ki table** ka screenshot Claude ko bhejo.
Agle upgrades: (1) demo app ko v2 models se jodna, (2) toxicity ko 22 ADMET properties tak badhana, (3) top molecules ko **Boltz-2** (2025 ka open-source structure + affinity model) se double-check karna.